# Reservoir Sampling

Reservoir sampling (Vitter, 1985) is a family of algorithms for choosing a **simple random sample of k items from a stream of unknown or very large length**, using only **one pass** and **O(k) memory**.

The classic variant, **Algorithm R**, works as follows:

1. Fill a "reservoir" array with the first k items from the stream.
2. For each subsequent item at position i (0-indexed), generate a random integer j in [0, i]. If j < k, replace reservoir[j] with the new item.
3. When the stream is exhausted, the reservoir contains a uniform random sample.

At every point in the algorithm, each of the n items seen so far has an equal probability **k/n** of being in the reservoir. This makes it ideal for sampling from data that doesn't fit in memory, arrives online, or whose total size isn't known in advance.

## Implementation

A straightforward pure-Python implementation using only the `random` module. We provide two functions: `reservoir_sample` returns the final sample, and `reservoir_sample_trace` yields the reservoir state after each element for pedagogical tracing.

In [ ]:
import random


def reservoir_sample(stream, k):
    """Algorithm R: select k items uniformly at random from stream."""
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
        else:
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = item
    return reservoir


def reservoir_sample_trace(stream, k):
    """Like reservoir_sample, but yields (index, item, slot, replaced, state) at each step."""
    reservoir = []
    for i, item in enumerate(stream):
        if i < k:
            reservoir.append(item)
            yield i, item, None, None, list(reservoir)
        else:
            j = random.randint(0, i)
            if j < k:
                replaced = reservoir[j]
                reservoir[j] = item
                yield i, item, j, replaced, list(reservoir)
            else:
                yield i, item, None, None, list(reservoir)

## Example: Step by Step

Let's trace the algorithm on a small stream of letters A through J, sampling k=3 items.

In [ ]:
random.seed(42)
stream = list('ABCDEFGHIJ')
k = 3

print(f"Stream: {stream},  k = {k}")
print()

final = None
for i, item, slot, replaced, state in reservoir_sample_trace(stream, k):
    if i < k:
        print(f"i={i}  item='{item}'  action=fill slot {i}                  reservoir={state}")
    elif slot is not None:
        print(f"i={i}  item='{item}'  action=replace slot {slot} (was '{replaced}')     reservoir={state}")
    else:
        print(f"i={i}  item='{item}'  action=skip (j >= k)              reservoir={state}")
    final = state

print(f"\nFinal sample: {final}")

Stream: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],  k = 3

i=0  item='A'  action=fill slot 0                  reservoir=['A']
i=1  item='B'  action=fill slot 1                  reservoir=['A', 'B']
i=2  item='C'  action=fill slot 2                  reservoir=['A', 'B', 'C']
i=3  item='D'  action=replace slot 0 (was 'A')     reservoir=['D', 'B', 'C']
i=4  item='E'  action=replace slot 0 (was 'D')     reservoir=['E', 'B', 'C']
i=5  item='F'  action=skip (j >= k)              reservoir=['E', 'B', 'C']
i=6  item='G'  action=replace slot 2 (was 'C')     reservoir=['E', 'B', 'G']
i=7  item='H'  action=skip (j >= k)              reservoir=['E', 'B', 'G']
i=8  item='I'  action=skip (j >= k)              reservoir=['E', 'B', 'G']
i=9  item='J'  action=replace slot 2 (was 'G')     reservoir=['E', 'B', 'J']

Final sample: ['E', 'B', 'J']


In [ ]:
random.seed(0)
stream_items = list(range(1, 21))
k = 5
trials = 100_000

counts = {x: 0 for x in stream_items}
for _ in range(trials):
    sample = reservoir_sample(iter(stream_items), k)
    for x in sample:
        counts[x] += 1

expected = trials * k / len(stream_items)
print(f"Stream: {stream_items}")
print(f"k = {k}, trials = {trials:,}, expected count per item = {expected:,.0f}")
print()
for x in stream_items:
    freq = counts[x] / trials
    bar = '#' * int(freq * 200)
    print(f"  Item {x:>2d}: {counts[x]:>6d} ({freq:.4f})  {bar}")

Stream: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
k = 5, trials = 100,000, expected count per item = 25,000

  Item  1:  24831 (0.2483)  #################################################
  Item  2:  25053 (0.2505)  ##################################################
  Item  3:  24883 (0.2488)  #################################################
  Item  4:  25150 (0.2515)  ##################################################
  Item  5:  24969 (0.2497)  #################################################
  Item  6:  25014 (0.2501)  ##################################################
  Item  7:  24935 (0.2493)  #################################################
  Item  8:  24943 (0.2494)  #################################################
  Item  9:  25158 (0.2516)  ##################################################
  Item 10:  25214 (0.2521)  ##################################################
  Item 11:  25237 (0.2524)  ################################################

## Longer Example

The real power of reservoir sampling is that it works on streams that don't fit in memory. Here we sample from a generator producing 1,000,000 items — only k items are ever stored.

In [ ]:
random.seed(123)

def big_stream(n):
    """A generator that yields n items one at a time — never all in memory."""
    for i in range(n):
        yield i

sample = reservoir_sample(big_stream(1_000_000), 10)
print(f"Sampled 10 items from a stream of 1,000,000:")
print(sample)
print(f"Reservoir size: {len(sample)} (O(k) memory, regardless of stream length)")

Sampled 10 items from a stream of 1,000,000:
[724144, 903923, 939324, 368224, 484155, 756560, 755086, 132963, 542869, 355425]
Reservoir size: 10 (O(k) memory, regardless of stream length)


## How It Works — Recap

1. **Fill phase.** The first k items go directly into the reservoir.
2. **Replace phase.** For the i-th item (i >= k), pick a random index j uniformly from [0, i]. If j < k, replace reservoir[j] with the new item; otherwise discard it.
3. **Termination.** When the stream ends after n items, the reservoir holds a uniform random sample of size k.

**Why is each item equally likely to be in the final sample?**

By induction. After seeing i items, each item in the reservoir is there with probability k/i. When item i+1 arrives:

- It enters the reservoir with probability k/(i+1).
- Each existing reservoir item survives (is not replaced) with probability 1 - 1/(i+1) = i/(i+1).
- So each old item's probability becomes (k/i) * (i/(i+1)) = k/(i+1).

All items end up with the same probability k/n. The algorithm uses O(k) memory, makes one pass, and requires only O(1) work per element.

## References

- Vitter, J.S. (1985). **"Random Sampling with a Reservoir."** *ACM Transactions on Mathematical Software*, 11(1), 37–57. [https://doi.org/10.1145/3147.3165](https://doi.org/10.1145/3147.3165)
- [Wikipedia: Reservoir sampling](https://en.wikipedia.org/wiki/Reservoir_sampling)